# Transcrição de vídeos com MoviePy e OpenAI Whisper

Este notebook prototipa uma pipeline simples para extrair o áudio de vídeos com MoviePy, transcrever o conteúdo com a biblioteca `openai-whisper` e salvar uma linha do tempo textual com timestamps em segundos.

O foco é prototipagem leve: poucas dependências, código direto, artefatos fáceis de inspecionar e saída em JSON/CSV para integração posterior com os demais experimentos de vídeo deste repositório.

## 1) Visão geral da abordagem

A pipeline é dividida em quatro etapas:

1. Receber um caminho local de vídeo.
2. Validar se o arquivo existe e se a extensão é suportada.
3. Ler o vídeo com `MoviePy`.
4. Extrair o áudio para um arquivo WAV mono em 16 kHz.
5. Enviar o áudio para o modelo Whisper local.
6. Salvar os segmentos retornados pelo Whisper com `start_sec`, `end_sec`, `duration_sec` e `text`.

O Whisper já retorna segmentos temporais com início e fim em segundos. Por isso, este notebook não tenta calcular timestamps manualmente a partir de frames do vídeo.

Para manter o protótipo leve, o modelo padrão é `base`. Se a máquina estiver lenta, troque para `tiny`. Se a qualidade estiver insuficiente, teste `small`.

## 2) Instalação das dependências

Execute esta célula apenas uma vez por ambiente. Depois da instalação, reinicie o kernel se o notebook pedir ou se algum pacote recém-instalado não for reconhecido.

Observação importante: a biblioteca oficial `openai-whisper` tradicionalmente funciona melhor em ambientes Python 3.8 a 3.11. Como este repositório usa Python 3.13 no `pyproject.toml`, o caminho mais previsível para este protótipo é criar um kernel separado com Python 3.11 e abrir este notebook nele.

Também é necessário ter `ffmpeg` disponível no sistema. O MoviePy e o Whisper usam FFmpeg para manipular áudio e vídeo.

In [1]:
# Execute somente se o ambiente ainda não tiver as dependências.
# Em um kernel Python 3.11, esta célula costuma ser suficiente para prototipagem local.

# %pip install moviepy openai-whisper pandas ipykernel

Looking in indexes: https://pytorch.org
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3) Importações e checagem rápida do ambiente

Esta célula importa as bibliotecas principais e mostra informações úteis para diagnóstico: versão do Python, disponibilidade de GPU via PyTorch e presença do executável `ffmpeg` no `PATH`.

Se `ffmpeg` aparecer como `None`, instale o FFmpeg antes de processar vídeos. No Windows, uma forma comum é instalar via Winget ou Chocolatey e reabrir o terminal/notebook para atualizar o `PATH`.

In [1]:
from __future__ import annotations

import json
import platform
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import pandas as pd
import torch
import whisper
from moviepy import VideoFileClip

print(f"Python: {sys.version.split()[0]}")
print(f"Plataforma: {platform.platform()}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
print(f"XPU disponível: {torch.xpu.is_available() if hasattr(torch, 'xpu') else False}")
print(f"ffmpeg: {shutil.which('ffmpeg')}")

Python: 3.13.8
Plataforma: Windows-11-10.0.26200-SP0
CUDA disponível: False
XPU disponível: False
ffmpeg: C:\Users\LuizAlbertodeAndrade\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg.Essentials_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.1-essentials_build\bin\ffmpeg.EXE


## 4) Configuração do notebook

Aqui ficam os caminhos e parâmetros principais. O notebook assume que será executado a partir da raiz do repositório ou diretamente de dentro da pasta `concepts_video`.

Ajuste `VIDEO_SOURCE` para apontar para o vídeo local que deseja transcrever. Os artefatos serão gravados em `concepts_video/outputs/transcriptions`.

Parâmetros principais:

- `VIDEO_SOURCE`: caminho local do vídeo.
- `MODEL_NAME`: modelo Whisper usado na transcrição. Use `tiny` para maior velocidade, `base` para equilíbrio inicial e `small` para mais qualidade.
- `LANGUAGE`: idioma esperado do áudio. Para vídeos em português, use `pt`. Para detecção automática, use `None`.
- `KEEP_EXTRACTED_AUDIO`: mantém o WAV intermediário para auditoria e reuso. Se quiser economizar espaço, troque para `False`.

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    """Resolve a raiz do repositório mesmo quando o notebook roda dentro de concepts_video."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "concepts_video").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
CONCEPTS_DIR = PROJECT_ROOT / "concepts_video"
INPUT_DIR = CONCEPTS_DIR / "input_videos"
OUTPUT_DIR = CONCEPTS_DIR / "outputs" / "transcriptions"
AUDIO_DIR = OUTPUT_DIR / "audio"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_SOURCE: str | Path = INPUT_DIR / "abuse_with_audio.mp4"

MODEL_NAME = "base"
LANGUAGE = "pt"
KEEP_EXTRACTED_AUDIO = True

AUDIO_SAMPLE_RATE = 16_000
AUDIO_NBYTES = 2
AUDIO_CODEC = "pcm_s16le"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"VIDEO_SOURCE: {VIDEO_SOURCE}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

PROJECT_ROOT: C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot
VIDEO_SOURCE: C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\input_videos\abuse_with_audio.mp4
OUTPUT_DIR: C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\outputs\transcriptions


## 5) Inspeção dos vídeos disponíveis

Esta célula lista os vídeos encontrados em `concepts_video/input_videos`. Ela é apenas uma ajuda de notebook para confirmar o nome do arquivo antes de ajustar `VIDEO_SOURCE`.

Caso você já tenha vídeos em outra pasta, pode apontar `VIDEO_SOURCE` diretamente para o caminho desejado.

In [4]:
VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}

available_videos = sorted(
    path for path in INPUT_DIR.glob("**/*")
    if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
)

local_videos_df = pd.DataFrame(
    {
        "video_path": [str(path.relative_to(PROJECT_ROOT)) for path in available_videos],
        "size_mb": [round(path.stat().st_size / (1024 * 1024), 2) for path in available_videos],
    }
)

local_videos_df

,video_path,size_mb
0,concepts_video\input_videos\abuse_with_audio.mp4,2.46


## 6) Utilitários de extração e serialização

As funções abaixo mantêm a pipeline organizada:

- `resolve_local_video_path`: transforma `VIDEO_SOURCE` em um caminho local absoluto e validado.
- `probe_video`: lê metadados básicos do vídeo, como duração e presença de áudio.
- `extract_audio_with_moviepy`: extrai o áudio para WAV mono em 16 kHz.
- `normalize_whisper_segments`: converte os segmentos do Whisper para um formato estável, com timestamps em segundos.
- `write_srt`: cria um arquivo `.srt` simples, útil para revisão manual em players de vídeo.

O áudio intermediário é intencionalmente separado da transcrição. Isso facilita depurar problemas de áudio antes de culpar o modelo de transcrição.

In [5]:
@dataclass(frozen=True)
class VideoMetadata:
    path: Path
    duration_sec: float | None
    fps: float | None
    width: int | None
    height: int | None
    has_audio: bool


def probe_video(video_path: Path) -> VideoMetadata:
    if not video_path.exists():
        raise FileNotFoundError(f"Vídeo não encontrado: {video_path}")

    with VideoFileClip(str(video_path), audio=True) as clip:
        width, height = clip.size if clip.size else (None, None)
        return VideoMetadata(
            path=video_path,
            duration_sec=float(clip.duration) if clip.duration is not None else None,
            fps=float(clip.fps) if clip.fps is not None else None,
            width=width,
            height=height,
            has_audio=clip.audio is not None,
        )


def safe_stem(path: Path) -> str:
    return "".join(char if char.isalnum() or char in {"-", "_"} else "_" for char in path.stem)

def resolve_local_video_path(source: str | Path) -> Path:
    path = Path(source).expanduser()
    if not path.is_absolute():
        project_candidate = (PROJECT_ROOT / path).resolve()
        cwd_candidate = (Path.cwd() / path).resolve()
        path = project_candidate if project_candidate.exists() else cwd_candidate

    path = path.resolve()
    if not path.exists():
        raise FileNotFoundError(f"Vídeo local não encontrado: {path}")
    if path.suffix.lower() not in VIDEO_EXTENSIONS:
        raise ValueError(f"Extensão de vídeo não suportada neste notebook: {path.suffix}")
    return path

def extract_audio_with_moviepy(video_path: Path, audio_dir: Path) -> Path:
    metadata = probe_video(video_path)
    if not metadata.has_audio:
        raise ValueError(f"O vídeo não possui faixa de áudio: {video_path}")

    audio_path = audio_dir / f"{safe_stem(video_path)}_mono_16khz.wav"

    with VideoFileClip(str(video_path), audio=True) as clip:
        if clip.audio is None:
            raise ValueError(f"O vídeo não possui faixa de áudio: {video_path}")

        clip.audio.write_audiofile(
            str(audio_path),
            fps=AUDIO_SAMPLE_RATE,
            nbytes=AUDIO_NBYTES,
            codec=AUDIO_CODEC,
            ffmpeg_params=["-ac", "1"],
            logger=None,
        )

    return audio_path


def normalize_whisper_segments(result: dict) -> list[dict]:
    segments = []
    for segment in result.get("segments", []):
        start_sec = float(segment["start"])
        end_sec = float(segment["end"])
        segments.append(
            {
                "id": int(segment.get("id", len(segments))),
                "start_sec": round(start_sec, 3),
                "end_sec": round(end_sec, 3),
                "duration_sec": round(max(0.0, end_sec - start_sec), 3),
                "text": segment.get("text", "").strip(),
            }
        )
    return segments


def seconds_to_srt_time(value: float) -> str:
    milliseconds = int(round(value * 1000))
    hours, remainder = divmod(milliseconds, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    seconds, milliseconds = divmod(remainder, 1000)
    return f"{hours:02}:{minutes:02}:{seconds:02},{milliseconds:03}"


def write_srt(segments: Iterable[dict], srt_path: Path) -> None:
    lines = []
    for index, segment in enumerate(segments, start=1):
        lines.extend(
            [
                str(index),
                f"{seconds_to_srt_time(segment['start_sec'])} --> {seconds_to_srt_time(segment['end_sec'])}",
                segment["text"],
                "",
            ]
        )
    srt_path.write_text("\n".join(lines), encoding="utf-8")

## 7) Carregamento do modelo Whisper

Esta célula carrega o modelo localmente. Na primeira execução, o Whisper pode baixar os pesos do modelo escolhido. Depois disso, os pesos ficam em cache no ambiente do usuário.

O uso de `fp16` só é ativado quando há CUDA disponível. Em CPU, `fp16=False` evita avisos e problemas de compatibilidade.

In [6]:
FP16 = torch.xpu.is_available() or torch.cuda.is_available()

model = whisper.load_model(MODEL_NAME)

print(f"Modelo carregado: {MODEL_NAME}")
print(f"fp16: {FP16}")

Modelo carregado: base
fp16: False


## 8) Função principal de transcrição

`transcribe_video` executa a pipeline inteira para um vídeo local:

1. Resolve e valida o caminho do arquivo.
2. Mede o vídeo local.
3. Extrai o áudio com MoviePy.
4. Transcreve o áudio com Whisper.
5. Normaliza os segmentos temporais.
6. Salva JSON, CSV e SRT.

O JSON é o artefato principal para integração programática. O CSV facilita inspeção em notebook. O SRT ajuda na validação visual junto com o vídeo original.

In [7]:
def transcribe_video(video_source: str | Path) -> dict:
    video_path = resolve_local_video_path(video_source)
    metadata = probe_video(video_path)
    audio_path = extract_audio_with_moviepy(video_path, AUDIO_DIR)

    result = model.transcribe(
        str(audio_path),
        language=LANGUAGE,
        task="transcribe",
        fp16=FP16,
        verbose=False,
    )

    segments = normalize_whisper_segments(result)
    output_stem = safe_stem(video_path)
    json_path = OUTPUT_DIR / f"{output_stem}_transcription.json"
    csv_path = OUTPUT_DIR / f"{output_stem}_segments.csv"
    srt_path = OUTPUT_DIR / f"{output_stem}.srt"

    payload = {
        "source": str(video_source),
        "video": str(video_path),
        "audio": str(audio_path) if KEEP_EXTRACTED_AUDIO else None,
        "model": MODEL_NAME,
        "language": result.get("language"),
        "duration_sec": round(metadata.duration_sec, 3) if metadata.duration_sec is not None else None,
        "fps": round(metadata.fps, 3) if metadata.fps is not None else None,
        "width": metadata.width,
        "height": metadata.height,
        "text": result.get("text", "").strip(),
        "segments": segments,
        "outputs": {
            "json": str(json_path),
            "csv": str(csv_path),
            "srt": str(srt_path),
        },
    }

    json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    pd.DataFrame(segments).to_csv(csv_path, index=False, encoding="utf-8")
    write_srt(segments, srt_path)

    if not KEEP_EXTRACTED_AUDIO:
        audio_path.unlink(missing_ok=True)

    return payload

## 9) Execução para um vídeo

Antes de rodar, confirme que `VIDEO_SOURCE` aponta para um arquivo local existente. Se estiver usando a pasta sugerida, coloque o vídeo em `concepts_video/input_videos` e atualize o nome em `VIDEO_SOURCE`.

A saída exibida abaixo mostra os caminhos dos artefatos e uma prévia dos segmentos transcritos.

In [8]:
payload = transcribe_video(VIDEO_SOURCE)

print("Artefatos gerados:")
print(f"- source: {payload['source']}")
print(f"- video: {payload['video']}")
for name, path in payload["outputs"].items():
    print(f"- {name}: {path}")

segments_df = pd.DataFrame(payload["segments"])
segments_df.head(20)

100%|██████████| 1001/1001 [00:01<00:00, 614.18frames/s]

Artefatos gerados:
- source: C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\input_videos\abuse_with_audio.mp4
- video: C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\input_videos\abuse_with_audio.mp4
- json: C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\outputs\transcriptions\abuse_with_audio_transcription.json
- csv: C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\outputs\transcriptions\abuse_with_audio_segments.csv
- srt: C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\outputs\transcriptions\abuse_with_audio.srt


,id,start_sec,end_sec,duration_sec,text
0,0,0.00,5.88,5.88,"Eu não sei mais o que fazer, ele me bate e diz..."
1,1,5.88,9.12,3.24,"Eu tenho medo, isso já acontece há muito tempo."


## 10) Execução em lote

Depois de validar um único vídeo, use esta célula para processar todos os vídeos encontrados em `concepts_video/input_videos`.

A execução em lote continua mesmo se um vídeo falhar. O resumo final mostra quais arquivos foram processados e quais retornaram erro.

In [ ]:
def transcribe_video_batch(video_sources: Iterable[str | Path]) -> pd.DataFrame:
    rows = []
    for video_source in video_sources:
        try:
            batch_payload = transcribe_video(video_source)
            rows.append(
                {
                    "source": str(video_source),
                    "video": batch_payload["video"],
                    "status": "ok",
                    "segments": len(batch_payload["segments"]),
                    "duration_sec": batch_payload["duration_sec"],
                    "json": batch_payload["outputs"]["json"],
                    "error": None,
                }
            )
        except Exception as exc:
            rows.append(
                {
                    "source": str(video_source),
                    "video": None,
                    "status": "error",
                    "segments": 0,
                    "duration_sec": None,
                    "json": None,
                    "error": str(exc),
                }
            )
    return pd.DataFrame(rows)


batch_sources = [*available_videos]

# Descomente para processar todos os vídeos locais encontrados na pasta de entrada.
# batch_df = transcribe_video_batch(batch_sources)
# batch_df

## 11) Leitura de uma transcrição salva

Esta célula ajuda a reabrir um JSON gerado anteriormente sem precisar rodar o Whisper de novo. Isso é útil para revisar timestamps, montar gráficos ou integrar a transcrição com outros sinais do vídeo.

In [ ]:
def load_transcription(json_path: Path) -> dict:
    return json.loads(json_path.read_text(encoding="utf-8"))


# Exemplo de uso depois de gerar uma transcrição:
# saved_payload = load_transcription(Path(payload["outputs"]["json"]))
# pd.DataFrame(saved_payload["segments"]).head(20)

## 12) Próximos ajustes possíveis

Este protótipo foi mantido propositalmente simples. Depois da validação inicial, os próximos incrementos mais úteis são:

- Ativar `word_timestamps=True` no `model.transcribe` quando timestamps por palavra forem necessários.
- Adicionar um campo `speaker` no JSON caso seja feita diarização com outra ferramenta.
- Integrar os segmentos de fala com os notebooks de postura/expressão usando `start_sec` e `end_sec` como eixo temporal comum.
- Criar uma célula de visualização com barras horizontais por segmento para revisar lacunas e sobreposição com eventos visuais.

Para a primeira versão, a saída por segmento já é suficiente para sincronizar fala com análises visuais feitas em outros notebooks deste projeto.